# Authors
- Raphael Wäspi ()
- Nicolas Huber (16-936-205)

## II - Clustering Coefficient - Coding Exercise (6 points)

For the provided datasets (you can find the dataset files in the Materials/Session 03 folder on OLAT):

1. Plot the average degree of the nearest neighbours knn as a function of the vertices degree.
2. Super impose the same quantity for the randomised network.
Hint: The randomised networks are obtained via multiple edge swaps via the networkx function
nx.algorithms.smallworld.random_reference. Make sure to set the parameter connectivity
= False to have faster execution. If execution is still slow, we provide randomised datasets.)
3. Compute the assortativity coefficient of the real network.
4. Compute the assortativity coefficient of the randomised network.

In [ ]:
# Imports
import match as m
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import os
import glob
import re
from tqdm import tqdm
from scipy.stats import pearsonr, spearmanr, kendalltau

In [ ]:
# Define colors with reduced opacity from a colormap
COLOR_BLUE = 'dodgerblue'
COLOR_RED = 'lightcoral'

# Define max number of k-neighbors
MAX_K = 11

### 1. Plot the average degree of the nearest neighbours knn as a function of the vertices degree.

In [ ]:
# Define the directory containing the .gml files
directory = 'assignment_1_data'

# Use glob to get a list of all .gml files in the directory
gml_files = glob.glob(os.path.join(directory, '*.gml'))

# Init an empty dictionary for all graphs - the key is the actual name, and the value is the graph data
graphs = {}

# Use tqdm to show a progress bar in Jupyter Notebook
with tqdm(total=len(gml_files), desc="Loading GML Files") as pbar:
    # Loop through the list of .gml files and load them
    for file_path in gml_files:
        with open(file_path, 'r') as file:
            content = file.read()
            
            # Use regex to extract the desired part
            match = re.search(r'/([^/]+)\.gml$', file_path)
            if match:
                graph_name = match.group(1)
                
                # Add to the graph dictionary
                graph = nx.read_gml(file_path)
                graphs[graph_name] = graph
                
                # Print the graph name when each part is finished
                print(f"Finished loading {graph_name}")
                
        pbar.update(1)  # Update the progress bar

In [ ]:
results = dict.fromkeys(graphs)
for key, value in results.items():
    results[key] = {}

In [ ]:
print(results)

In [ ]:
num_graphs = len(graphs)

# Create a grid of subplots with rows for each k value and columns for each graph
fig, axs = plt.subplots(MAX_K - 1, num_graphs, figsize=(90, 15))

for k in range(1, MAX_K):
    for i, (graph_name, g) in enumerate(graphs.items()):
        average_knn_degrees = []
        node_degrees = []
        
        # Iterate over all nodes and extract neighbors
        
        for node in g.nodes():
            neighbors = list(g.neighbors(node))
            # calculate degrees for all neighbors and also the average knn degree
            if len(neighbors) >= k:
                # x-value 
                node_degrees.append(len(neighbors))
                
                # y-value
                knn_degrees = [g.degree(neighbor) for neighbor in neighbors]
                average_knn_degree = np.mean(knn_degrees)
                average_knn_degrees.append(average_knn_degree) 
                
        
        
        # Append to result dict
        results[graph_name][f"Average {k}-NN Degree"] = average_knn_degrees
        results[graph_name][f"Average {k}-NN Degree (x)"] = node_degrees
        
        # Plot on the corresponding subplot
        axs[k - 1, i].scatter(x=node_degrees, y=average_knn_degrees, color=COLOR_BLUE)
        axs[k - 1, i].set_xlabel("Vertex Degree")
        axs[k - 1, i].set_ylabel(f"Average {k}-NN Degree")
        axs[k - 1, i].set_title(f"Average {k}-NN Degree vs. Vertex Degree: {graph_name}")
        axs[k - 1, i].grid(True)
 
# Adjust spacing between subplots
plt.title(label="Average degree of the nearest neighbours knn as a function of the vertices degree", loc="center")
plt.tight_layout()
plt.show()

## 2. Super impose the same quantity for the randomised network.

Hint: The randomised networks are obtained via multiple edge swaps via the networkx function
nx.algorithms.smallworld.random_reference. Make sure to set the parameter connectivity
= False to have faster execution. If execution is still slow, we provide randomised datasets.)

In [ ]:
random_graphs = graphs.copy()
seed = 42

for graph_name, g in random_graphs.items():
    random_graphs[graph_name] = nx.algorithms.smallworld.random_reference(G=g, niter=1, connectivity=False, seed=seed)
    

In [ ]:
num_graphs = len(random_graphs)

# Create a grid of subplots with rows for each k value and columns for each graph
fig, axs = plt.subplots(MAX_K - 1, num_graphs, figsize=(90, 15))

for k in range(1, MAX_K):
    for i, (graph_name, g) in enumerate(random_graphs.items()):
        average_knn_degrees = []
        node_degrees = []
        
        # Iterate over all nodes and extract neighbors
        
        for node in g.nodes():
            neighbors = list(g.neighbors(node))
            # calculate degrees for all neighbors and also the average knn degree
            if len(neighbors) >= k:
                # x-value 
                node_degrees.append(len(neighbors))
                
                # y-value
                knn_degrees = [g.degree(neighbor) for neighbor in neighbors]
                average_knn_degree = np.mean(knn_degrees)
                average_knn_degrees.append(average_knn_degree) 
                
        
        
        # Append to result dict
        results[graph_name][f"Average {k}-NN Degree (Randomized)"] = average_knn_degrees
        results[graph_name][f"Average {k}-NN Degree (Randomized) (x)"] = node_degrees
        
        # Plot on the corresponding subplot
        axs[k - 1, i].scatter(x=node_degrees, y=average_knn_degrees, color=COLOR_BLUE)
        axs[k - 1, i].set_xlabel("Vertex Degree")
        axs[k - 1, i].set_ylabel(f"Average {k}-NN Degree")
        axs[k - 1, i].set_title(f"Average {k}-NN Degree vs. Vertex Degree: {graph_name}")
        axs[k - 1, i].grid(True)
 
# Adjust spacing between subplots
plt.title(label="Average degree of the nearest neighbours knn as a function of the vertices degree (Randomized)", loc="center")
plt.tight_layout()
plt.show()

In [ ]:
# Create a grid of subplots with rows for each k value and columns for each graph
fig, axs = plt.subplots(MAX_K - 1, num_graphs, figsize=(90, 15))

for k in range(1, MAX_K):
    for i, (graph_name, g) in enumerate(random_graphs.items()):
        
        # Plot on the corresponding subplot
        axs[k - 1, i].scatter(x=results[graph_name][f"Average {k}-NN Degree (x)"], y=results[graph_name][f"Average {k}-NN Degree"], label='normal', color=COLOR_BLUE) # Normal
        axs[k - 1, i].scatter(x=results[graph_name][f"Average {k}-NN Degree (Randomized) (x)"], y=results[graph_name][f"Average {k}-NN Degree (Randomized)"], label='randomized', color=COLOR_RED, alpha=0.35) # Randomized
        axs[k - 1, i].set_xlabel("Vertex Degree")
        axs[k - 1, i].set_ylabel(f"Average {k}-NN Degree")
        axs[k - 1, i].set_title(f"Average {k}-NN Degree vs. Vertex Degree: {graph_name}")
        axs[k - 1, i].grid(True)
 
# Adjust spacing between subplots
plt.title(label="Average degree of the nearest neighbours knn as a function of the vertices degree (Normal vs. Randomization)", loc="center")
plt.tight_layout()
plt.show()

## 3. Compute the assortativity coefficient of the real network.

In [ ]:
for i, (graph_name, g) in enumerate(graphs.items()):
    # Compute degree_assortativity_coefficient
    assortativity_coefficient = nx.degree_assortativity_coefficient(G=g)
    results[graph_name]["Assortativity Coefficient"] = assortativity_coefficient
    print(f"The network {graph_name} has an Assortativity Coefficient of {assortativity_coefficient:10.4f}.")

## 4. Compute the assortativity coefficient of the randomised network.

In [ ]:
for i, (graph_name, g) in enumerate(random_graphs.items()):
    # Compute degree_assortativity_coefficient
    assortativity_coefficient = nx.degree_assortativity_coefficient(G=g)
    results[graph_name]["Assortativity Coefficient (Randomized)"] = assortativity_coefficient
    print(f"The network {graph_name} has an Assortativity Coefficient of {assortativity_coefficient:10.4f}.")

In [ ]:
print("Comparison of the assortativity coefficient in normal vs. randomized networks")
for graph_name, res in results.items():
    print(f"{graph_name} - Normal: {res['Assortativity Coefficient']:10.4f} Randomized: {res['Assortativity Coefficient (Randomized)']:10.4f}")

- The assortativity coefficient ranges from -1 to 1:

- A positive coefficient (closer to 1) indicates assortative mixing, meaning nodes with similar degrees tend to connect.
- A negative coefficient (closer to -1) indicates disassortative mixing, meaning nodes with dissimilar degrees tend to connect.
- A coefficient near 0 suggests a network with little to no assortative or disassortative mixing, indicating a more random or heterogeneous connection pattern.

## IV - Centralities - Coding Exercise (9 points)

1. Compute degree, closeness, betweenness and eigenvector centrality for each node.
2. Do a scatter plot of each pair of centrality (6 plots total). Compute Pearson, Spearman and Kendall correlation coefficient for each pair and note them on the scatter plots. Hint: Centralities measures are available on NetworkX, while correlation coefficients are available on the module scipy.stats.
3. Repeat point 2 on randomised versions of the graphs provided. Hint: Check point 2 of Exercise 2, for indications on the randomisation function to use in NetworkX.
4. Briefly explain, for the Jazz collaboration network from Gleiser and Danon [2003], what each centrality measure means in practice.

In [ ]:
results_4 = dict.fromkeys(graphs)
for key, value in results_4.items():
    results_4[key] = {}

In [ ]:
# find smallest graph (development only)
for name, g in graphs.items():
    print(len(g.nodes()), name)

In [ ]:
# this will take quite some time
for i, (graph_name, g) in enumerate(graphs.items()):
    if True #if graph_name in ["graph_jazz_collab", "graph_game_thrones"]: # if True:
        node_properties = {}
        node_properties["Degree"] = nx.degree_centrality(G=g)
        node_properties["Closeness"] = nx.closeness_centrality(G=g)
        node_properties["Betweenness"] = nx.betweenness_centrality(G=g)
        node_properties["Eigenvector Centrality"] = nx.eigenvector_centrality(G=g)
    else:
        # only for development, remove if before submission
        pass
    
    results_4[graph_name] = node_properties

In [ ]:
# Define CENTRALITY_MEASURES and MAX_K
CENTRALITY_MEASURES = ["Degree", "Closeness", "Betweenness", "Eigenvector Centrality"]


# Create a grid of subplots with rows for each correlation measure and columns for each centrality pair
fig, axs = plt.subplots(len(CENTRALITY_MEASURES), len(CENTRALITY_MEASURES), figsize=(15, 15))

# Iterate through centrality pairs
for i in range(len(CENTRALITY_MEASURES)):
    for j in range(len(CENTRALITY_MEASURES)):
        if i != j:  # Avoid plotting the same centrality against itself
            x_centrality = CENTRALITY_MEASURES[i]
            y_centrality = CENTRALITY_MEASURES[j]

            for graph_name, node_properties in results_4.items():
                if graph_name in ["graph_jazz_collab", "graph_game_thrones"]:
                    x_data = list(node_properties[x_centrality])
                    y_data = list(node_properties[y_centrality])

                    # Determine subplot position
                    row = i
                    col = j

                    # Create a scatter plot in the corresponding subplot
                    axs[row, col].scatter(x_data, y_data, color='blue', alpha=0.5)
                    axs[row, col].set_xlabel(f"{x_centrality}")
                    axs[row, col].set_ylabel(f"{y_centrality}")
                    axs[row, col].set_title(f"{x_centrality} vs. {y_centrality} - {graph_name}")
                    axs[row, col].grid(True)

                    # Calculate Pearson correlation coefficient
                    pearson_corr, _ = pearsonr(x_data, y_data)

                    # Calculate Spearman correlation coefficient
                    spearman_corr, _ = spearmanr(x_data, y_data)

                    # Calculate Kendall correlation coefficient
                    kendall_corr, _ = kendalltau(x_data, y_data)

                    # Note the correlation coefficients on the scatter plot
                    corr_text = f"Pearson: {pearson_corr:.4f}\nSpearman: {spearman_corr:.4f}\nKendall: {kendall_corr:.4f}"
                    axs[row, col].text(0.1, 0.9, corr_text, transform=axs[row, col].transAxes)

# Adjust subplot layout
plt.suptitle("Centrality Measures Scatter Plots and Correlation Coefficients", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust the suptitle position
plt.show()